# 02 — Baseline YOLOv8-seg Training

**DRISHTI** — AI-powered marine debris detection from side-scan sonar

Fine-tunes YOLOv8s-seg from COCO-pretrained weights on the Watertank sonar debris dataset.
This is the primary detection + segmentation model for the pipeline.

**Run on:** Colab (free T4) or Kaggle (free T4/P100). ⚠ Requires GPU runtime!

In [ ]:
# ---- Environment setup ----
!pip install -q ultralytics opencv-python-headless h5py albumentations

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('⚠ No GPU detected! Change Runtime → T4 GPU in Colab.')

In [ ]:
import os
import sys
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

# Paths
WORK_DIR = Path('drishti_training')
WORK_DIR.mkdir(exist_ok=True)
DATA_DIR = WORK_DIR / 'data'
DATA_DIR.mkdir(exist_ok=True)

CLASSES = [
    'bottle', 'can', 'chain', 'drink_carton', 'hook',
    'propeller', 'shampoo_bottle', 'standing_bottle',
    'tire', 'valve', 'wall', 'net'
]
NC = len(CLASSES)
print(f'{NC} classes: {CLASSES}')

## 1. Prepare Dataset

Download Watertank HDF5 → extract images + masks → convert to YOLO-seg format → split.

In [ ]:
# Download dataset
HDF5_URL = (
    'https://github.com/mvaldenegro/marine-debris-fls-datasets/'
    'releases/download/watertank-v1.0/watertank_segmentation.h5'
)
HDF5_PATH = DATA_DIR / 'watertank_segmentation.h5'

if not HDF5_PATH.exists():
    print('Downloading Watertank dataset...')
    !wget -q -O {HDF5_PATH} {HDF5_URL}
    print('Done.')

import h5py
hf = h5py.File(HDF5_PATH, 'r')
images = hf['images'][:] if 'images' in hf else hf[list(hf.keys())[0]][:]
masks = hf['masks'][:] if 'masks' in hf else hf[list(hf.keys())[1]][:]
hf.close()

print(f'Loaded {len(images)} images, shape: {images.shape}')
print(f'Masks shape: {masks.shape}, unique values: {np.unique(masks)}')

In [ ]:
# ---- Convert masks to YOLO-seg polygon format ----

# Watertank mask pixel values → YOLO class IDs (0-indexed)
MASK_TO_YOLO = {1:0, 2:1, 3:2, 4:3, 5:4, 6:5, 7:6, 8:7, 9:8, 10:9, 11:10}

def mask_to_yolo_polygons(mask, img_w, img_h):
    """Convert a multi-class pixel mask to YOLO-seg label lines."""
    lines = []
    for mask_val, yolo_cls in MASK_TO_YOLO.items():
        binary = (mask == mask_val).astype(np.uint8) * 255
        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for cnt in contours:
            if cv2.contourArea(cnt) < 30:
                continue
            eps = 0.005 * cv2.arcLength(cnt, True)
            approx = cv2.approxPolyDP(cnt, eps, True)
            if len(approx) < 3:
                continue
            pts = approx.squeeze()
            if pts.ndim == 1:
                continue
            coords = []
            for x, y in pts:
                coords.extend([f'{max(0,min(1,x/img_w)):.6f}', f'{max(0,min(1,y/img_h)):.6f}'])
            lines.append(f'{yolo_cls} ' + ' '.join(coords))
    return lines

print('Conversion function ready.')

In [ ]:
import random
random.seed(42)
np.random.seed(42)

# Create splits
indices = list(range(len(images)))
random.shuffle(indices)

n = len(indices)
n_train = int(n * 0.8)
n_val = int(n * 0.1)

splits = {
    'train': indices[:n_train],
    'val': indices[n_train:n_train + n_val],
    'test': indices[n_train + n_val:],  # held-out, REAL-ONLY
}

for split_name, split_idx in splits.items():
    img_dir = DATA_DIR / 'splits' / split_name / 'images'
    lbl_dir = DATA_DIR / 'splits' / split_name / 'labels'
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)
    
    for idx in split_idx:
        img = images[idx]
        msk = masks[idx]
        
        # Handle shape
        if img.ndim == 3 and img.shape[0] in (1, 3):
            img = img[0]
        if msk.ndim == 3:
            msk = msk[0] if msk.shape[0] == 1 else msk[:,:,0]
        
        # Normalize to uint8
        if img.dtype != np.uint8:
            if img.max() <= 1.0:
                img = (img * 255).astype(np.uint8)
            else:
                img = np.clip(img, 0, 255).astype(np.uint8)
        if msk.dtype != np.uint8:
            msk = msk.astype(np.uint8)
        
        h, w = img.shape[:2]
        stem = f'wt_{idx:05d}'
        
        # Save image (convert grayscale to 3-channel for YOLO)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
        cv2.imwrite(str(img_dir / f'{stem}.png'), img_rgb)
        
        # Convert and save labels
        labels = mask_to_yolo_polygons(msk, w, h)
        with open(lbl_dir / f'{stem}.txt', 'w') as f:
            f.write('\n'.join(labels))
            if labels:
                f.write('\n')
    
    print(f'{split_name}: {len(split_idx)} images')

print(f'\n✓ Dataset prepared at {DATA_DIR / "splits"}')

In [ ]:
# ---- Write YOLO data config ----
yaml_content = f"""# DRISHTI YOLOv8-seg dataset config
path: {(DATA_DIR / 'splits').resolve()}
train: train/images
val: val/images
test: test/images

task: segment

nc: {NC}
names: {CLASSES}
"""

yaml_path = WORK_DIR / 'drishti.yaml'
yaml_path.write_text(yaml_content)
print(f'Data config written to: {yaml_path}')
print(yaml_content)

## 2. Train YOLOv8s-seg

Transfer learning from COCO-pretrained weights. `patience=20` enables early stopping.

In [ ]:
# ---- Load pretrained model ----
model = YOLO('yolov8s-seg.pt')  # COCO-pretrained, ~23MB
print(f'Model loaded: yolov8s-seg.pt')
print(f'Parameters: {sum(p.numel() for p in model.model.parameters()):,}')

In [ ]:
# ---- Train ----
results = model.train(
    data=str(yaml_path),
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,          # early stop if val doesn't improve for 20 epochs
    device=0,             # GPU
    project=str(WORK_DIR / 'runs'),
    name='yolov8s_seg_baseline',
    exist_ok=True,
    pretrained=True,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    warmup_epochs=3,
    weight_decay=0.0005,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    flipud=0.2,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    save=True,
    save_period=10,
    val=True,
    plots=True,
)

## 3. Training Curves

In [ ]:
import pandas as pd

# Load training results CSV
results_dir = WORK_DIR / 'runs' / 'yolov8s_seg_baseline'
csv_path = results_dir / 'results.csv'

if csv_path.exists():
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    # Loss curves
    for col, ax, title in [
        ('train/box_loss', axes[0,0], 'Box Loss'),
        ('train/seg_loss', axes[0,1], 'Seg Loss'),
        ('train/cls_loss', axes[0,2], 'Cls Loss'),
    ]:
        if col in df.columns:
            ax.plot(df['epoch'], df[col], 'b-', label='Train', alpha=0.8)
            val_col = col.replace('train/', 'val/')
            if val_col in df.columns:
                ax.plot(df['epoch'], df[val_col], 'r-', label='Val', alpha=0.8)
            ax.set_xlabel('Epoch')
            ax.set_title(title)
            ax.legend()
            ax.grid(True, alpha=0.3)
    
    # Metrics curves
    metric_cols = [
        ('metrics/precision(B)', axes[1,0], 'Precision'),
        ('metrics/recall(B)', axes[1,1], 'Recall'),
        ('metrics/mAP50(B)', axes[1,2], 'mAP50'),
    ]
    for col, ax, title in metric_cols:
        if col in df.columns:
            ax.plot(df['epoch'], df[col], 'g-', linewidth=2)
            ax.set_xlabel('Epoch')
            ax.set_title(title)
            ax.set_ylim(0, 1)
            ax.grid(True, alpha=0.3)
    
    plt.suptitle('YOLOv8s-seg Training Curves', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Print final metrics
    last = df.iloc[-1]
    print(f"\nFinal metrics (epoch {int(last.get('epoch', -1))}):\n")
    for col in df.columns:
        if 'metrics' in col or 'mAP' in col:
            print(f"  {col}: {last[col]:.4f}")
else:
    print('No results CSV found — training may still be running.')

## 4. Validation on Held-Out Test Set

This is the **honest** number — real images the model never saw during training.

In [ ]:
# ---- Validate on test set ----
best_model_path = results_dir / 'weights' / 'best.pt'

if best_model_path.exists():
    best_model = YOLO(str(best_model_path))
    
    val_results = best_model.val(
        data=str(yaml_path),
        split='test',
        imgsz=640,
        batch=16,
        plots=True,
        verbose=True,
    )
    
    print('\n' + '=' * 50)
    print('HELD-OUT TEST SET RESULTS (pitch deck numbers)')
    print('=' * 50)
    if hasattr(val_results, 'box'):
        print(f'Detection mAP50:    {val_results.box.map50:.4f}')
        print(f'Detection mAP50-95: {val_results.box.map:.4f}')
    if hasattr(val_results, 'seg'):
        print(f'Segment mAP50:      {val_results.seg.map50:.4f}')
        print(f'Segment mAP50-95:   {val_results.seg.map:.4f}')
else:
    print(f'Best model not found at {best_model_path}')

## 5. Visual Inference on Test Images

In [ ]:
# ---- Run inference on test images and visualise ----
test_img_dir = DATA_DIR / 'splits' / 'test' / 'images'
test_images = sorted(test_img_dir.glob('*.png'))[:8]

if best_model_path.exists() and test_images:
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    axes = axes.flatten()
    
    for i, img_path in enumerate(test_images):
        results = best_model.predict(
            str(img_path), imgsz=640, conf=0.25, verbose=False
        )
        
        # Plot with detections
        annotated = results[0].plot()
        axes[i].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        
        n_det = len(results[0].boxes) if results[0].boxes is not None else 0
        axes[i].set_title(f'{img_path.stem} ({n_det} det)', fontsize=9)
        axes[i].axis('off')
    
    plt.suptitle('YOLOv8s-seg Predictions on Test Set', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('Model or test images not available.')

## 6. Next Steps

1. **`03_synthetic_data_generation.ipynb`** — generate synthetic nets/pipes to boost rare classes
2. **`04_confidence_calibration.ipynb`** — Platt scaling for honest confidence scores
3. **`05_error_analysis.ipynb`** — investigate false positives (rock clusters)
4. **`06_shadow_geometry_filter.ipynb`** — geometric FP suppression
5. **`07_export_and_benchmark.ipynb`** — ONNX + INT8 + benchmarks

In [ ]:
print('Baseline training complete!')
print(f'Best model: {best_model_path}')
print('Proceed to 03_synthetic_data_generation.ipynb')